# Plan de test 0.4.0 — notebook de vérification

Déroule [le plan de test manuel](../docs/test_plan_0.4.0.md) sur une **vraie course**
(portail Al Kamel — réseau requis). Chaque cellule imprime des contrôles ✅/❌
automatiques ; les critères de *plausibilité* (nombre d'arrêts vs le rapport de
course, récit des changements de leader, …) restent à juger à l'œil — voir le plan.

> Par défaut le notebook charge les 6h de Spa 2019 (connu pour marcher). Pour la
> validation complète, visez une course de **24h** (Le Mans) : nuit pour
> `add_day_night`, beaucoup d'arrêts, batailles, changements de leader.

In [ ]:
from pathlib import Path

import pandas as pd

import endurancepy as ep
from endurancepy import plotting

ep.set_log_level("INFO")  # progression découverte/téléchargements sur stderr
Path("./cache").mkdir(exist_ok=True)
ep.Cache.enable_cache("./cache")


def check(label: str, ok) -> None:
    print(("✅" if bool(ok) else "❌"), label)

## 1. Charger la course

Choisir la course ici — `ep.list_seasons(SERIES)` donne les ids de saison
(format `"NN_YYYY"`, visible aussi dans l'URL du portail).

In [ ]:
# print(ep.list_seasons("WEC"))  # décommenter pour repérer l'id de saison

YEAR, SERIES, EVENT, SEASON = 2019, "WEC", "Spa", "08_2018-2019"
# Pour la validation complète (nuit, ~30 arrêts/voiture, batailles) :
# YEAR, SERIES, EVENT, SEASON = 2023, "WEC", "Le Mans", "<id via list_seasons>"

session = ep.get_session(YEAR, SERIES, EVENT, "Race")
session.load(season=SEASON)
laps = session.laps
print(f"{len(laps)} tours, {len(session.cars)} voitures")

## 2. Parser : `PitTime` et `Hour`

Nouveau en 0.4.0 : le temps passé aux stands (`PitTime`) est conservé sur les
in-laps, et `Hour` (heure du jour, en heures décimales) est rempli depuis la
colonne `HOUR` du CSV.

In [ ]:
check(
    "PitTime seulement sur les in-laps",
    laps.loc[laps["PitInTime"].isna(), "PitTime"].isna().all(),
)
check(
    "PitTime renseigné sur des in-laps",
    laps.loc[laps["PitInTime"].notna(), "PitTime"].notna().any(),
)
check("Hour rempli (heure du jour)", laps["Hour"].notna().any())
print(f"Hour couvre {laps['Hour'].min():.2f}h -> {laps['Hour'].max():.2f}h")
laps.loc[
    laps["PitInTime"].notna(), ["CarNumber", "LapNumber", "LapTime", "PitTime", "Hour"]
].head()

## 3. `pit_stops` — arrêts + signal undercut/overcut

Une ligne par arrêt. `PosBefore` (tour précédent) vs `PosAfter` (`settle` tours
après) ; `PlacesGained = PosBefore - PosAfter` (positif = places gagnées).
À juger : nombre d'arrêts/voiture plausible (≈ 25–35 sur 24h), `PitTime`
réaliste (≈ 1–2 min au Mans, ravitaillement inclus).

In [ ]:
stops = ep.pit_stops(session)
check("au moins un arrêt détecté", len(stops) > 0)
check("trié par tour", stops["Lap"].is_monotonic_increasing)
both = stops.dropna(subset=["PosBefore", "PosAfter", "PlacesGained"])
check(
    "PlacesGained = PosBefore - PosAfter",
    (both["PosBefore"] - both["PosAfter"] == both["PlacesGained"]).all(),
)
print(stops.groupby("CarNumber").size().rename("Stops").describe())
stops.head(10)

In [ ]:
# La fenêtre `settle` change PosAfter mais doit rester du même ordre.
for settle in (1, 2, 3):
    table = ep.pit_stops(session, settle=settle)
    print(f"settle={settle}: PlacesGained moyen = {table['PlacesGained'].mean()}")

In [ ]:
# Bulle par arrêt : x = tour, y = voiture, taille = temps aux stands, couleur = classe.
plotting.plot_pit_stops(session)

## 4. `stint_summary` — relais et dégradation

`Degradation` = pente des temps propres en s/tour (positif = la voiture ralentit).
À juger : longueur de relais ≈ fenêtre carburant (Hypercar au Mans : ~10–14 tours),
pentes petites et plutôt positives sur les relais longs et propres.

In [ ]:
stints = ep.stint_summary(session)
expected = laps.dropna(subset=["Stint"]).groupby(["CarNumber", "Stint"]).ngroups
check("une ligne par (voiture, relais)", len(stints) == expected)
check("StartLap <= EndLap", (stints["StartLap"] <= stints["EndLap"]).all())
paced = stints.dropna(subset=["BestLap", "MedianLap"])
check("BestLap <= MedianLap", (paced["BestLap"] <= paced["MedianLap"]).all())
stints[stints["CarNumber"] == session.cars[0]]

## 5. `driver_summary` — pilotes

À juger : 2–3 pilotes par voiture, la somme des `TimeInCar` d'un équipage
approche la durée de course (moins les arrêts), `Consistency` en dixièmes/secondes.

In [ ]:
drivers = ep.driver_summary(session)
crewed = set(laps.dropna(subset=["Driver"])["CarNumber"].astype("string"))
check("chaque voiture a au moins un pilote", set(drivers["CarNumber"]) == crewed)
check("Consistency >= 0", (drivers["Consistency"].dropna() >= 0).all())
print("Temps en voiture par équipage :")
print(
    drivers.groupby("CarNumber")["TimeInCar"].sum().sort_values(ascending=False).head()
)
drivers

## 6. `lead_changes` — périodes de leadership

Une ligne par série de tours en tête ; les transitions sont les changements de
leader. À juger : la succession colle au récit connu de la course.

In [ ]:
overall = ep.lead_changes(session)
led_rows = laps[laps["Position"] == 1].dropna(subset=["LapNumber"])
check("tous les tours en tête couverts", overall["Laps"].sum() == len(led_rows))
winner = str(session.results.iloc[0]["CarNumber"])
check("dernier leader = vainqueur", str(overall.iloc[-1]["Leader"]) == winner)
overall

In [ ]:
# Par classe : le dernier leader de chaque classe doit gagner sa classe.
ep.lead_changes(session, in_class=True)

## 7. `battles` — batailles en piste

Paires de voitures adjacentes dans l'ordre et sous `within` d'écart pendant
`min_laps` tours consécutifs (robuste aux échanges de position). À juger : les
plus longues batailles correspondent à des duels réellement observés.

In [ ]:
WITHIN = "2s"
fights = ep.battles(session, within=WITHIN)
if len(fights):
    check(
        "MinGap <= MeanGap <= seuil",
        (
            (fights["MinGap"] <= fights["MeanGap"])
            & (fights["MeanGap"] <= pd.Timedelta(WITHIN))
        ).all(),
    )
    check(
        "Laps = ToLap - FromLap + 1",
        (fights["Laps"] == fights["ToLap"] - fights["FromLap"] + 1).all(),
    )
else:
    print(f"Aucune bataille sous {WITHIN} — élargir `within`.")
check(
    "élargir le seuil ne réduit pas le nombre de batailles",
    len(ep.battles(session, within="10s")) >= len(fights),
)
fights.sort_values("Laps", ascending=False).head(10)

## 8. `time_lost` — temps perdu (proxy trafic/erreurs)

Somme, par voiture, des dépassements de sa propre médiane sur tours propres au
vert. À juger : les voitures de pointe perdent moins que le milieu de peloton ;
une course pleine de neutralisations gonfle le signal (attendu).

In [ ]:
lost = ep.time_lost(session)
check("TimeLost >= 0", (lost["TimeLost"] >= pd.Timedelta(0)).all())
check(
    "un seuil plus élevé réduit le total",
    ep.time_lost(session, threshold="2s")["TimeLost"].sum() <= lost["TimeLost"].sum(),
)
lost.sort_values("TimeLost", ascending=False).head(10)

## 9. `fuel_corrected` — correction carburant

Ramène chaque tour à la charge de carburant du début de relais (`rate` s/tour
d'essence brûlée). Sur un relais propre, la tendance corrigée est plus plate —
il reste la dégradation pneus et le trafic.

In [ ]:
RATE = 0.03  # s/tour — ordre de grandeur usuel
corrected = ep.fuel_corrected(session, rate=RATE)
delta = corrected - laps["LapTime"]
stint_start = laps.groupby(["CarNumber", "Stint"])["LapNumber"].transform("min")
first_lap = (laps["LapNumber"] == stint_start) & laps["LapTime"].notna()
check(
    "correction nulle au 1er tour de relais",
    (delta[first_lap] == pd.Timedelta(0)).all(),
)
check("correction jamais négative", (delta.dropna() >= pd.Timedelta(0)).all())
check(
    "rate=0 rend LapTime inchangé",
    ep.fuel_corrected(session, rate=0.0).equals(laps["LapTime"]),
)

# Le premier relais de la voiture de tête, brut vs corrigé (en secondes) :
one = laps[(laps["CarNumber"] == session.cars[0]) & (laps["Stint"] == 1)]
pd.DataFrame(
    {
        "LapNumber": one["LapNumber"],
        "Brut (s)": one["LapTime"].dt.total_seconds(),
        "Corrigé (s)": corrected[one.index].dt.total_seconds(),
    }
)

## 10. `add_day_night` — bandes nocturnes

Sur une course de 24h : bande(s) ≈ 22h–6h locale, qui ne masquent pas les
courbes et cohabitent avec `add_track_status`. Sur une course diurne : **0 bande**
(no-op) — c'est le résultat attendu avec le Spa 2019 par défaut.

In [ ]:
fig = plotting.plot_lap_evolution(session)
before = len(fig.layout.shapes or ())
plotting.add_day_night(fig, session)
night_bands = len(fig.layout.shapes or ()) - before
print(f"{night_bands} bande(s) nocturne(s) — attendu : >= 1 sur 24h, 0 en plein jour")
plotting.add_track_status(fig, session)  # combo 0.3.0 + 0.4.0
fig

## 11. Cas limites

Les tables acceptent `Session`, `Laps` ou un sous-ensemble filtré, et rendent
des DataFrames vides bien typés quand il n'y a rien à dire.

In [ ]:
first_class = laps["Class"].dropna().iloc[0]
subset = ep.pit_stops(laps.pick_classes(first_class))
check("sous-ensemble filtré accepté", (subset["Class"] == first_class).all())

empty = ep.pit_stops(pd.DataFrame())
check(
    "entrée vide -> table vide typée",
    empty.empty and str(empty["PitTime"].dtype) == "timedelta64[ns]",
)
check("battles sur une entrée vide", ep.battles(pd.DataFrame()).empty)
check("lead_changes sur une entrée vide", ep.lead_changes(pd.DataFrame()).empty)

## Bilan

Si tous les ✅ passent et que les critères « à juger » collent à la réalité de la
course (rapport de course, replay du timing), la 0.4.0 est validée. Reste la
release : voir [`RELEASING.md`](../RELEASING.md) (tag `v0.4.0` après merge sur `main`).